# Cross-feeding in a two-species community

_Investigation `community-cross-feeding` — coder reproduction notebook._

**Question.** Can CRM-constrained dynamic FBA, composed as a multi-species bigraph on a shared nutrient pool, reproduce niche differentiation by cross-feeding rather than competitive exclusion?

A one-study showcase of the CRM-FBA workspace's most impressive composite: a two-species E. coli community cross-feeding on a shared glucose/acetate pool.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/Users/eranagmon/code/CRM_FBA--demo-investigation').is_dir():
    REPO = Path('/Users/eranagmon/code/CRM_FBA--demo-investigation')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from crm_dfba.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Two-species cross-feeding community (`two-species-community`)

**Question.** When a glucose specialist and an acetate specialist share a single glucose/acetate pool with no spatial separation, does CRM-constrained dynamic FBA reproduce niche differentiation by cross-feeding — i.e. does the glucose specialist's acetate overflow sustain distinct, temporally offset growth of the acetate specialist — rather than simple competitive exclusion?

**Objective.** Simulate the two-species crm_fba_community composite on the E. coli core GSM and measure per-species biomass and the shared glucose/acetate pools to determine whether the acetate specialist's growth is sustained by the glucose specialist's acetate overflow (niche differentiation) rather than by direct competition for glucose.

**Hypothesis.** The glucose specialist (high glucose Vmax, tight O2 cap that forces acetate overflow) blooms first and draws glucose to zero while secreting acetate into the shared pool; the acetate specialist (high acetate Vmax, low glucose affinity) then does the majority of its growth on that byproduct, producing two temporally offset growth phases — the signature of cross-feeding-driven niche partitioning.

**Claim.** The glucose specialist's acetate overflow sustains a distinct acetate specialist: 93% of the acetate specialist's biomass gain occurs after the acetate pool peaks, and glucose is exhausted exactly at the peak (t=8.3) — niche differentiation by cross-feeding, not competitive exclusion.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `crm_fba_community` | `crm_dfba.composites.crm_fba_community` | 360 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `crm_dfba.composites.crm_fba_community`** — `spec_crm_dfba_composites_crm_fba_community` (a plain, editable dict)


_composite spec file for `crm_dfba.composites.crm_fba_community` not found under `crm_dfba/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: two-species-community ===
STUDY = 'two-species-community'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Runtime knobs — edit freely. STEPS = number of composite steps;
# INTERVAL = global dt filling ${interval} placeholders (a per-process
# interval pinned in the edit cell above takes precedence).
STEPS_crm_fba_community = 360
INTERVAL_crm_fba_community = 0.1

if RERUN:
    with quiet():  # the sim prints per-step progress; keep it out of the notebook
        # Generic process-bigraph protocol (no workspace runner detected):
        from viva_superpowers.composite_spec import build_composite_from_spec
        comp = build_composite_from_spec(spec_crm_dfba_composites_crm_fba_community, {'interval': INTERVAL_crm_fba_community}, core=core)
        comp.run(STEPS_crm_fba_community)  # writes the composite's declared emitter
    print(f'ran 1 simulation(s) -> {RUNS_DB}')
else:
    print("RERUN=False — rendering committed", RUNS_DB)

### Visualizations

_Results are shown by the figures below, produced by the run above._


**community_cross_feeding**


In [ ]:
# community_cross_feeding
show_viz(_render_one('local:CommunityCrossFeeding', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| glucose-drawn-to-zero | kind=final_value path=substrates.glucose window=whole_run | op at_most high 0.05 provenance {'kind': 'theory', 'note': 'A closed batch with active uptake exhausts the limiting sugar.'} |
| acetate-transient-overflow-peak | kind=peak_value path=substrates.acetate window=whole_run | op at_least low 5.0 provenance {'kind': 'theory', 'note': 'O2-capped glucose oxidation secretes acetate as overflow.'} |
| acetate-specialist-grows-post-peak | kind=fraction_of_growth_after_peak path=biomass_acetate_specialist window=whole_run | op at_least low 0.5 provenance {'kind': 'theory', 'note': 'Cross-feeding (not competition) means growth follows byproduct availability.'} |


## Open decisions
- Run the acetate-uptake-knockout negative control?
